<a href="https://colab.research.google.com/github/dcapsan/opt-practica-5/blob/main/Practica_2_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

# Instalar compatibilidad de Keras
!pip install -q tf-keras pillow numpy

import tensorflow as tf
import tf_keras as keras
from PIL import Image, ImageOps
import numpy as np

# 1. Cargar el modelo exportado y las etiquetas
model = keras.models.load_model('keras_model.h5', compile=False)
class_names = open('labels.txt', 'r').readlines()

# Función para predecir cada foto
def analizar_foto(ruta_imagen):
    data = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)
    image = Image.open(ruta_imagen).convert('RGB')
    size = (224, 224)
    image = ImageOps.fit(image, size, Image.Resampling.LANCZOS)
    image_array = np.asarray(image)

    # Normalización al rango [-1, 1]
    normalized_image_array = (image_array.astype(np.float32) / 127.5) - 1
    data[0] = normalized_image_array

    prediction = model.predict(data, verbose=0)
    index = np.argmax(prediction)
    clase = class_names[index].strip()
    confianza = prediction[0][index]

    print(f"--- Imagen: {ruta_imagen} ---")
    print(f"Predicción: {clase}")
    print(f"Nivel de confianza: {confianza * 100:.2f}%\n")

# 2. Ejecutar las 3 imágenes
analizar_foto('clara.jpeg')
analizar_foto('ambigua.png')
analizar_foto('ajena.png')

--- Imagen: clara.jpeg ---
Predicción: 0 Cascos
Nivel de confianza: 98.51%

--- Imagen: ambigua.png ---
Predicción: 0 Cascos
Nivel de confianza: 79.75%

--- Imagen: ajena.png ---
Predicción: 0 Cascos
Nivel de confianza: 99.71%



--- Imagen: clara.jpeg ---
Predicción: 0 Cascos
Nivel de confianza: 98.51%

--- Imagen: ambigua.png ---
Predicción: 0 Cascos
Nivel de confianza: 79.75%

--- Imagen: ajena.png ---
Predicción: 0 Cascos
Nivel de confianza: 99.71%

¿Cómo se usa el modelo exportado?
Se cargan los pesos del modelo (keras_model.h5) en Python usando tf_keras. Las imágenes de entrada se redimensionan a 224x224 píxeles y se normalizan sus valores de color entre -1 y 1. La función model.predict() pasa la imagen por la red neuronal y devuelve la probabilidad asignada a cada etiqueta de labels.txt (altavoz o cascos).

¿Cómo se comporta?
Con la imagen clara: Acierta la categoría real (cascos) con una confianza muy alta (habitualmente superior al 90%), ya que los patrones visuales coinciden de forma evidente con el entrenamiento.

Con la imagen ambigua: Muestra dudas y la probabilidad se reparte de forma más igualada entre ambas opciones, tirando más hacia los cascos, al compartir rasgos físicos como rejillas o formas redondeadas.

Con la imagen ajena: El modelo no tiene una opción de "desconocido" o "ninguna de las anteriores", por lo que se ve obligado a clasificarla como altavoz o como cascos al 100%. Suele asignarla basándose en parecidos casuales de color, fondo o iluminación, a menudo con una confianza engañosamente alta.